# Unit 4 SQL Practice: Social Media Advertisement Performance Dataset
These exercises are aligned to WJEC-style Unit 4 SQL skills and run against a local SQLite copy of the social-media advertisement dataset.

The prompts intentionally mention data types (`INT`, `REAL`, `TEXT`, `DATETIME`) so learners connect query strategy to the kind of value being filtered, grouped, or updated.



In [ ]:
from pathlib import Path
import sys

notebooks_root = Path.cwd()
while not (notebooks_root / "helpers").exists() and notebooks_root != notebooks_root.parent:
    notebooks_root = notebooks_root.parent

sys.path.insert(0, str(notebooks_root))

from helpers import setup_social_media_ads_sql_notebook

setup_social_media_ads_sql_notebook()



### Exercise 1: Premium Campaign Budgets (SELECT, WHERE, ORDER BY)
Write a query to show `campaign_id`, `name`, and `total_budget` for campaigns with `total_budget` greater than `90000`.
Order the output by `total_budget` in descending order.
Why query this data type: `total_budget` is `REAL`, so threshold filtering helps identify expensive campaigns for finance review.



In [ ]:
%%sql
SELECT campaign_id, name, total_budget
FROM campaigns
WHERE total_budget > 90000
ORDER BY total_budget DESC;


### Exercise 2: Long And Expensive Campaigns (SELECT, WHERE, AND, ORDER BY)
Write a query to output `campaign_id`, `name`, `duration_days`, and `total_budget` where `duration_days` is at least `88` and `total_budget` is at least `70000`.
Order by `duration_days` descending, then `total_budget` descending.
Why query this data type: combining `INTEGER` duration and `REAL` budget helps evaluate cost over campaign lifetime.



In [ ]:
%%sql
SELECT campaign_id, name, duration_days, total_budget
FROM campaigns
WHERE duration_days >= 88 AND total_budget >= 70000
ORDER BY duration_days DESC, total_budget DESC;


### Exercise 3: Narrow User Segment (SELECT, WHERE, IN, ORDER BY)
Write a query to display `user_id`, `country`, and `location` for users in age group `55-65` and country in `France` or `Japan`.
Order by `country` ascending and then `user_id` ascending.
Why query this data type: `TEXT` demographic fields are used for audience segmentation in ad targeting.



In [ ]:
%%sql
SELECT user_id, country, location
FROM users
WHERE age_group = '55-65' AND country IN ('France', 'Japan')
ORDER BY country ASC, user_id ASC;


### Exercise 4: Event Volume By Type (SELECT, GROUP BY, ORDER BY)
Write a query to output each `event_type` and its count as `event_count` from `ad_events`.
Group by `event_type` and order by `event_count` descending.
Why query this data type: categorical `TEXT` event labels are aggregated to compare funnel stages like impressions, clicks, and purchases.



In [ ]:
%%sql
SELECT event_type, COUNT(*) AS event_count
FROM ad_events
GROUP BY event_type
ORDER BY event_count DESC;


### Exercise 5: Ads In One Named Campaign (Subquery)
Use a subquery to list `ad_id`, `ad_type`, and `target_age_group` for ads belonging to campaign name `Campaign_12_Q3`.
Order by `ad_id` ascending.
Why query this data type: using a `TEXT` campaign name in a subquery mirrors business users searching by label rather than numeric IDs.



In [ ]:
%%sql
SELECT ad_id, ad_type, target_age_group
FROM ads
WHERE campaign_id = (
    SELECT campaign_id
    FROM campaigns
    WHERE name = 'Campaign_12_Q3'
)
ORDER BY ad_id ASC;


### Exercise 6: Engagement Mix For A Targeting Slice (JOIN, WHERE, GROUP BY)
Use a join between `ad_events` and `ads` to show `event_type` and count as `event_count` for ads where platform is `Instagram`, target gender is `Male`, and ad type is `Video`.
Group by `event_type` and order by `event_count` descending.
Why query this data type: joining event facts to ad metadata links behavioral data to targeting attributes for campaign analysis.



In [ ]:
%%sql
SELECT e.event_type, COUNT(*) AS event_count
FROM ad_events AS e
JOIN ads AS a ON e.ad_id = a.ad_id
WHERE a.ad_platform = 'Instagram' AND a.target_gender = 'Male' AND a.ad_type = 'Video'
GROUP BY e.event_type
ORDER BY event_count DESC;


### Exercise 7: Purchase Events By Country For One Ad (JOIN, WHERE, GROUP BY)
Use a join between `ad_events` and `users` to show each `country` and purchase count as `purchase_events` for ad `1` where event type is `Purchase`.
Group by `country` and order by `purchase_events` descending, then `country` ascending.
Why query this data type: this combines `INTEGER` identifiers with `TEXT` geography to localize conversion behavior.



In [ ]:
%%sql
SELECT u.country, COUNT(*) AS purchase_events
FROM ad_events AS e
JOIN users AS u ON e.user_id = u.user_id
WHERE e.ad_id = 1 AND e.event_type = 'Purchase'
GROUP BY u.country
ORDER BY purchase_events DESC, u.country ASC;


### Exercise 8: Create A Campaign Audit Table (CREATE TABLE)
Create a table named `CampaignAudit` with fields:
`auditID` as `INT NOT NULL PRIMARY KEY`, `campaignID` as `INT NOT NULL`, `actionType` as `CHAR(20) NOT NULL`, `actionDate` as `DATETIME NOT NULL`, and `notes` as `CHAR(80)`.
Why query this data type: audit logging needs stable key fields plus typed date/time columns for traceability.



In [ ]:
%%sql
CREATE TABLE CampaignAudit (
    auditID INT NOT NULL PRIMARY KEY,
    campaignID INT NOT NULL,
    actionType CHAR(20) NOT NULL,
    actionDate DATETIME NOT NULL,
    notes CHAR(80)
);


### Exercise 9: Insert One Audit Record (INSERT INTO)
Insert one row into `CampaignAudit` using values:
`1001`, `20`, `BudgetCheck`, `2025-01-15 09:00:00`, `Winter campaign budget reviewed`.
Why query this data type: inserting mixed numeric, text, and datetime values reinforces schema-aware data entry.



In [ ]:
%%sql
INSERT INTO CampaignAudit (auditID, campaignID, actionType, actionDate, notes)
VALUES (1001, 20, 'BudgetCheck', '2025-01-15 09:00:00', 'Winter campaign budget reviewed');


### Exercise 10: Update The Audit Note (UPDATE, WHERE)
Update `CampaignAudit` so `notes` becomes `Winter campaign budget review completed` for the row where `auditID` is `1001`.
Why query this data type: precise `WHERE` updates on key fields prevent accidental bulk edits in operational logs.



In [ ]:
%%sql
UPDATE CampaignAudit
SET notes = 'Winter campaign budget review completed'
WHERE auditID = 1001;
